# TestCompare

This notebook compares **ADMM MPC + LSTM Forecast**, **Single-Agent MPC + LSTM Forecast**, and **MADRL + Safety Projection** on the same projection-safe bundle-derived environment.

All three rollouts share the same agent/bus/battery/future-horizon configuration reconstructed from the projection-safe training bundle. The main remaining control differences are network coordination and controller family, not cfg drift.

The first run may spend noticeable time in `ensure_madrl_shared_data(...)` if the shared-data cache has not been precomputed yet.

In [4]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "configs").exists():
    repo_root = repo_root.parent
if not (repo_root / "configs").exists():
    raise RuntimeError("Could not locate the project root from the notebook working directory.")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from configs import compose_experiment_config
from scripts.utils import grid_notebook_workflow as grid_nb
from scripts.utils.admm_mpc_notebook_helpers import collect_admm_mpc_rollout
from scripts.utils.experiment_notebook_utils import resolve_madrl_model_root
from scripts.utils.grid_notebook_workflow import collect_madrl_rollout, load_training_run_bundle
from scripts.utils.madrl_shared_data import ensure_madrl_shared_data
from scripts.utils.project_paths import project_root as resolve_project_root


In [ ]:
PROJECT_ROOT = resolve_project_root()
DATA_DIR = PROJECT_ROOT / "data"

TEST_START_DATE = "2020-06-01"
TEST_END_DATE = "2020-06-02"
PREDICTION_MODE = "normal"
SHOW_PROGRESS = True

CHECKPOINT_ROOT = None
PROJECTION_MODEL_ROOT = None
MADRL_PROJECTION_LABEL = "MADRL + Safety Projection"
ADMM_LABEL = "ADMM MPC + LSTM Forecast"
SINGLE_AGENT_LABEL = "Single-Agent MPC + LSTM Forecast"

W_SOC_PEN = 2.0
EVAL_W_VOLTAGE_PEN = 10.0
EVAL_W_LINE_PEN = 10.0
EVAL_W_TRAFO_PEN = 10.0


In [6]:
resolved_projection_model_root = resolve_madrl_model_root(
    algorithm="MATD3_SAFE_POC",
    prediction_mode=PREDICTION_MODE,
    experiment_name="train_projection_safe",
    model_root=PROJECTION_MODEL_ROOT,
    root=PROJECT_ROOT,
    checkpoint_root=CHECKPOINT_ROOT,
)
reference_bundle = load_training_run_bundle(resolved_projection_model_root)
reference_experiment = reference_bundle["experiment_controls"]
reference_data = reference_bundle["data_controls"]
reference_battery = reference_bundle["battery_controls"]
reference_train = reference_bundle["train_controls"]

forecast_controls = dict(reference_experiment.get("forecast_controls") or {})
if forecast_controls:
    forecast_controls["auto_train_missing"] = False
resolved_prediction_mode = PREDICTION_MODE or reference_data["prediction_mode"]

cfg = compose_experiment_config(
    profile=reference_train.get("profile", "base"),
    algorithm="MATD3",
    model_family=reference_train.get("model_family", "mlp"),
    data_dir=DATA_DIR,
    device=reference_experiment.get("device_request"),
    runtime_mode=reference_experiment.get("runtime_mode", "performance"),
    seed=int(reference_experiment.get("seed", 0)),
    require_cuda=reference_experiment.get("require_cuda"),
)
grid_nb.apply_notebook_experiment_settings(
    cfg,
    prediction_mode=resolved_prediction_mode,
    test_start_date=TEST_START_DATE or reference_data.get("test_start_date"),
    test_end_date=TEST_END_DATE or reference_data.get("test_end_date"),
    agent_profiles=reference_data["agent_profiles"],
    agent_bus_ids=reference_data.get("agent_bus_ids"),
    load_scale=reference_data.get("load_scale"),
    pv_scale=reference_data.get("pv_scale"),
    battery_controls=reference_battery,
    forecast_controls=forecast_controls,
    future_horizon=reference_data.get("future_horizon"),
    train_year=reference_data.get("train_year"),
    test_year=reference_data.get("test_year"),
)
cfg.reward.w_soc_pen = W_SOC_PEN
cfg.reward.w_voltage_pen = EVAL_W_VOLTAGE_PEN
cfg.reward.w_line_pen = EVAL_W_LINE_PEN
cfg.reward.w_trafo_pen = EVAL_W_TRAFO_PEN
cfg.env.episode_limit = int(round(24.0 / cfg.env.dt))
if abs(float(cfg.env.dt) * int(cfg.env.episode_limit) - 24.0) > 1e-9:
    raise ValueError("TestCompare notebook expects one full day per episode.")

shared_data_result = ensure_madrl_shared_data(cfg)
cfg.runtime.shared_data_dir = str(shared_data_result.shared_data_dir)
cfg.runtime.shared_data_signature = str(shared_data_result.signature_hash)

display(
    pd.Series(
        {
            "test_start_date": cfg.data.test_start_date,
            "test_end_date": cfg.data.test_end_date,
            "prediction_mode": resolved_prediction_mode,
            "forecast_backend": cfg.forecast.type,
            "resolved_projection_model_root": str(resolved_projection_model_root),
            "import_price_adder_eur_per_kwh": float(cfg.reward.import_price_adder_eur_per_kwh),
            "export_subsidy_eur_per_kwh": float(cfg.reward.export_subsidy_eur_per_kwh),
            "seed": int(reference_experiment.get("seed", 0)),
            "future_horizon": int(cfg.env.future_horizon),
            "agent_profiles": list(reference_data["agent_profiles"]),
            "episode_limit": int(cfg.env.episode_limit),
            "shared_data_signature": cfg.runtime.shared_data_signature,
            "shared_data_reused": bool(shared_data_result.reused),
        },
        name="testcompare_config",
    )
)


ValueError: Managed LSTM forecast artifacts are missing or incompatible:
- signal='price', artifact='C:\Users\10856\Desktop\GithubProject\MADRL_ESS\artifacts\forecast\lstm\h24\price\price_lstm_h24.pt', 配置不一致: dropout(expected=0.1, actual=0.0), hidden_size(expected=128, actual=64), num_layers(expected=2, actual=1). 建议删除旧 artifact，或开启 cfg.forecast.auto_train_missing=True 自动重训。
- signal='pv', artifact='C:\Users\10856\Desktop\GithubProject\MADRL_ESS\artifacts\forecast\lstm\h24\pv\pv_lstm_h24.pt', 配置不一致: hidden_size(expected=96, actual=64). 建议删除旧 artifact，或开启 cfg.forecast.auto_train_missing=True 自动重训。

In [ ]:
print(f"Starting {ADMM_LABEL}...")
admm_mpc_rollout = collect_admm_mpc_rollout(
    cfg,
    prediction_mode=resolved_prediction_mode,
    show_progress=SHOW_PROGRESS,
)
print("Done:", admm_mpc_rollout.meta.get("controller", ADMM_LABEL))

print(f"Starting {SINGLE_AGENT_LABEL}...")
single_agent_mpc_rollout = grid_nb.collect_mpc_rollout(
    cfg,
    prediction_mode=resolved_prediction_mode,
    label=SINGLE_AGENT_LABEL,
)
print("Done:", single_agent_mpc_rollout.meta.get("controller", SINGLE_AGENT_LABEL))

print(f"Starting {MADRL_PROJECTION_LABEL}...")
madrl_projection = collect_madrl_rollout(
    cfg,
    model_root=resolved_projection_model_root,
    algorithm="MATD3_SAFE_POC",
    experiment_name="train_projection_safe",
    checkpoint_root=CHECKPOINT_ROOT,
    label=MADRL_PROJECTION_LABEL,
)
print("Done:", madrl_projection.meta.get("controller", MADRL_PROJECTION_LABEL))

rollouts = [admm_mpc_rollout, single_agent_mpc_rollout, madrl_projection]
metrics_df = grid_nb.compare_rollout_metrics(*rollouts)
economic_table = grid_nb.build_compare_economic_table(metrics_df)
safety_table = grid_nb.build_compare_safety_table(metrics_df)

display(
    pd.DataFrame(
        [
            {
                "controller": rollout.meta.get("controller"),
                "prediction_mode": rollout.meta.get("prediction_mode"),
                "forecast_backend": rollout.meta.get("forecast_backend"),
                "single_agent_mpc_price_mode": rollout.meta.get("single_agent_mpc_price_mode", "n/a"),
                "shared_data_signature": rollout.meta.get("shared_data_signature", cfg.runtime.shared_data_signature),
                "model_root": rollout.meta.get("model_root", str(resolved_projection_model_root) if rollout is madrl_projection else "n/a"),
            }
            for rollout in rollouts
        ]
    )
)
display(grid_nb.build_compare_warning_banner(*rollouts))
display(metrics_df)
display(economic_table)
display(safety_table)


In [ ]:
display(grid_nb.plot_price_prediction_comparison(*rollouts))
display(grid_nb.plot_power_balance_comparison(*rollouts))
display(grid_nb.plot_voltage_profile_comparison(*rollouts))
display(grid_nb.plot_net_load_comparison(*rollouts))
display(grid_nb.plot_battery_power_and_soc_comparison(*rollouts))
